# Aula 3.1 — Visualizar imagens e bounding boxes

**Bloco 3 — Matplotlib + Seaborn**  
**Projeto:** IC-FAPEMA — Detecção de capacetes em motociclistas via drone  
**Objetivo:** Exibir imagens e desenhar bounding boxes das detecções do YOLO

---

## 📖 Tabela de Termos

| Termo | Conceito | Função no código |
|-------|---------|------------------|
| **Matplotlib** | Biblioteca de visualização — gera gráficos e imagens | `import matplotlib.pyplot as plt` |
| **pyplot** | Módulo do Matplotlib para plotar | `plt.imshow()`, `plt.show()` |
| **imshow()** | Exibe uma imagem na tela | `plt.imshow(imagem)` |
| **figure()** | Cria uma janela/tela para o gráfico | `plt.figure(figsize=(10, 6))` |
| **subplots()** | Cria figura e eixo juntos | `fig, ax = plt.subplots(1, figsize=(10, 6))` |
| **Rectangle** | Forma geométrica para desenhar a bounding box | `patches.Rectangle((x, y), w, h)` |
| **add_patch()** | Adiciona a bounding box sobre a imagem | `ax.add_patch(retangulo)` |
| **patches** | Módulo do Matplotlib com formas geométricas | `from matplotlib import patches` |
| **axis('off')** | Esconde os eixos numéricos da figura | `ax.axis('off')` |

## 🧠 Conceito

Até agora você trabalhou com os **dados das detecções** (coordenadas, classes, estatísticas no Pandas).  
O Matplotlib serve para **ver** essas detecções — colocar as bounding boxes em cima da imagem real do drone.

O fluxo é:
1. Carregar a imagem (como array NumPy — você já sabe isso!)
2. Exibir a imagem com `imshow()`
3. Converter as coordenadas YOLO (0 a 1) para pixels
4. Desenhar um retângulo colorido em cima de cada detecção

> **Detalhe importante:** o YOLO guarda o **centro** da caixa.  
> Para desenhar o retângulo, você precisa do **canto superior esquerdo**:  
> `x_canto = (x_centro - largura/2) × largura_imagem`

In [ ]:
# Importar bibliotecas
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import patches   # módulo com formas geométricas (retângulo, círculo etc.)

## 💻 Código

In [ ]:
# ── 1. Criar uma imagem simulada do drone (fundo cinza) ──────────────────────
# np.ones cria array preenchido com 1s → multiplicamos por 200 para tom cinza
# shape: (altura_pixels, largura_pixels, canais_RGB)
imagem = np.ones((480, 640, 3), dtype=np.uint8) * 200

print("Shape da imagem:", imagem.shape)   # (480, 640, 3)

In [ ]:
# ── 2. Detecções do YOLO (formato normalizado 0 a 1) ─────────────────────────
# Cada detecção é um dicionário com classe e coordenadas normalizadas
# classe 0 = com capacete | classe 1 = sem capacete
deteccoes = [
    {'classe': 1, 'x_centro': 0.52, 'y_centro': 0.38, 'largura': 0.20, 'altura': 0.15},
    {'classe': 0, 'x_centro': 0.75, 'y_centro': 0.60, 'largura': 0.18, 'altura': 0.20},
    {'classe': 1, 'x_centro': 0.30, 'y_centro': 0.45, 'largura': 0.22, 'altura': 0.17},
]

In [ ]:
# ── 3. Dimensões reais da imagem em pixels ────────────────────────────────────
img_largura = imagem.shape[1]   # 640 pixels (eixo horizontal)
img_altura  = imagem.shape[0]   # 480 pixels (eixo vertical)

print(f"Largura: {img_largura}px | Altura: {img_altura}px")

In [ ]:
# ── 4. Criar a figura e exibir a imagem ──────────────────────────────────────
fig, ax = plt.subplots(1, figsize=(10, 6))   # cria figura (10x6 polegadas) e eixo
ax.imshow(imagem)                             # exibe a imagem no eixo

# ── 5. Desenhar cada bounding box sobre a imagem ─────────────────────────────
for det in deteccoes:

    # Converter coordenadas YOLO (centro normalizado) → canto superior esquerdo em pixels
    # YOLO guarda o centro; o retângulo precisa do canto superior esquerdo
    x_pixel = (det['x_centro'] - det['largura']  / 2) * img_largura
    y_pixel = (det['y_centro'] - det['altura']   / 2) * img_altura
    w_pixel =  det['largura']  * img_largura    # largura em pixels
    h_pixel =  det['altura']   * img_altura     # altura em pixels

    # Escolher cor: vermelho = sem capacete (1), verde = com capacete (0)
    cor = 'red' if det['classe'] == 1 else 'green'

    # Criar o retângulo da bounding box
    retangulo = patches.Rectangle(
        (x_pixel, y_pixel),   # ponto de origem: canto superior esquerdo
        w_pixel,               # largura do retângulo em pixels
        h_pixel,               # altura do retângulo em pixels
        linewidth=2,           # espessura da borda
        edgecolor=cor,         # cor da borda
        facecolor='none'       # interior transparente
    )

    # Adicionar o retângulo sobre a imagem
    ax.add_patch(retangulo)

    # Adicionar rótulo com a classe acima do retângulo
    label = 'sem capacete' if det['classe'] == 1 else 'com capacete'
    ax.text(x_pixel, y_pixel - 5, label, color=cor, fontsize=9, weight='bold')

# ── 6. Título e exibição final ────────────────────────────────────────────────
ax.set_title('Detecções do drone — Aula 3.1')   # título da figura
ax.axis('off')                                    # esconde os eixos numéricos
plt.tight_layout()                                # ajusta margens automaticamente
plt.show()                                        # exibe a figura

---
## 📋 Resumo da Aula 3.1

| O que aprendeu | Função no código | Para que serve no projeto |
|---|---|---|
| Criar figura e eixo | `plt.subplots()` | Base para exibir a imagem |
| Exibir imagem | `ax.imshow(imagem)` | Mostra o frame do drone |
| Converter coordenadas YOLO → pixels | `(x_centro - largura/2) × img_largura` | Posicionar a caixa corretamente |
| Desenhar bounding box | `patches.Rectangle()` + `ax.add_patch()` | Marcar cada detecção visualmente |
| Colorir por classe | `'red' if classe == 1 else 'green'` | Diferenciar infratores visualmente |